# PINN vs SSF: 60-Second Demo — No training required. Load pre-computed results and see the failures.

In [1]:
# Install in Colab
!git clone https://github.com/NodeBox-Labs/pinn-vs-ssf-schrodinger.git
%cd pinn-vs-ssf-schrodinger
!pip install -q -r requirements.txt

Cloning into 'pinn-vs-ssf-schrodinger'...
remote: Enumerating objects: 355, done.
remote: Counting objects: 100% (355/355), done.
remote: Compressing objects: 100% (246/246), done.
remote: Total 355 (delta 191), reused 245 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (355/355), 522.84 KiB | 9.86 MiB/s, done.
Resolving deltas: 100% (191/191), done.
/content/pinn-vs-ssf-schrodinger


In [2]:
from src.reference.split_step import reference_solution
import yaml

with open("configs/case_d3_two_soliton_collision.yaml") as f:
    cfg = yaml.safe_load(f)
ref = reference_solution(cfg)
print(f"Reference computed on {len(ref['x'])} points, {len(ref['t'])} snapshots.")

Reference computed on 1024 points, 101 snapshots.


In [ ]:
import urllib.request
import torch

# Download the pre-trained model from your release
url = "https://github.com/NodeBox-Labs/pinn-vs-ssf-schrodinger/releases/download/v1.0.0/d3_two_soliton_collision_seed42.pt"
filename = "d3_two_soliton_collision_seed42.pt"

print(f"Downloading {filename}...")
urllib.request.urlretrieve(url, filename)

# Load the model
ckpt = torch.load(filename, map_location="cpu", weights_only=False)
print("PINN checkpoint loaded. Case:", ckpt["metrics"]["case"])
print("Rel-L2 final:", ckpt["metrics"]["rel_l2_final"])

[d3_two_soliton_collision] generating reference ...
[d3_two_soliton_collision] training PINN (50306 params) ...
  Phase 1: Adam
Traceback (most recent call last):
  File "/content/pinn-vs-ssf-schrodinger/src/train.py", line 217, in <module>
    main(args.config, _parse_overrides(args.override))
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/pinn-vs-ssf-schrodinger/src/train.py", line 120, in main
    loss.backward()
    ~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        self, gradient, retain_graph, create_graph, inputs=inputs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/torch/autograd/__init__.py", line 381, in backward
    _engine_run_backward(
    ~~~~~~~~~~~~~~~~~~~~^
        tensors,
        ^^^^^^^^
    ...<5 lines>...
        accumulate_grad=True,
        ^

FileNotFoundError: [Errno 2] No such file or directory: 'results/checkpoints/d3_two_soliton_collision_seed42.pt'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t_idx = -1  # final time
x = ref["x"]
psi_ref = ref["psi"][t_idx]
psi_pinn = ckpt["preds"][t_idx]

plt.figure(figsize=(10, 4))
plt.plot(x, np.abs(psi_ref)**2, lw=2, label="SSF (reference)")
plt.plot(x, np.abs(psi_pinn)**2, "--", lw=2, label=f"PINN (rel-L2 = {ckpt['metrics']['rel_l2_final']:.2f})")
plt.xlabel("x"); plt.ylabel("|ψ|²")
plt.title("Two-soliton collision at T=8 — the PINN fails to resolve the peaks")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()